# Performance Internals — 04: Join Strategies Deep Dive

## What you will learn
Joins are the most expensive operation in most Spark jobs.
Choosing the wrong join strategy can make a job **10-100x slower**.

In this notebook you will:
1. Understand all 5 Spark join strategies and when each is used
2. Benchmark `BroadcastHashJoin` vs `SortMergeJoin` vs `ShuffledHashJoin`
3. Control join strategy explicitly with hints
4. Understand join ordering and how to help the optimizer
5. Handle skewed joins — the silent killer of Spark performance
6. Use AQE's dynamic join conversion

## The 5 Join Strategies

```
┌──────────────────────────────────────────────────────────────────┐
│ Strategy              │ Shuffle │ Sort │ When chosen              │
├──────────────────────────────────────────────────────────────────┤
│ BroadcastHashJoin     │  NO    │  NO  │ One side < threshold     │
│ ShuffledHashJoin      │  YES   │  NO  │ Medium tables, equi-join │
│ SortMergeJoin         │  YES   │  YES │ Large tables (default)   │
│ BroadcastNestedLoop   │  NO    │  NO  │ Non-equi join, tiny table│
│ CartesianProduct      │  NO    │  NO  │ CROSS JOIN (dangerous!)  │
└──────────────────────────────────────────────────────────────────┘

Decision order:
1. Can one side be broadcast?       → BroadcastHashJoin
2. Is there enough memory?          → ShuffledHashJoin
3. Default                          → SortMergeJoin
```


In [1]:
import os, time, random, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('join-strategies')
    .master(MASTER)
    .config('spark.executor.memory',            '2g')
    .config('spark.driver.memory',              '1g')
    .config('spark.executor.cores',             '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 15:33:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/08 15:33:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark 4.0.2


## Step 1 — Create Test Datasets

We create datasets of different sizes to benchmark join strategies.
The size ratio between tables determines which strategy is optimal.


In [ ]:
random.seed(42)

CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture","Toys"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR","IN","MX"]
STATUSES   = ["active","inactive","vip","blocked"]

def make_customers(n):
    return [(i, f"Customer_{i}", random.choice(COUNTRIES),
             random.choice(STATUSES), round(random.uniform(0, 10000), 2))
            for i in range(1, n+1)]

def make_products(n):
    return [(i, f"Product_{i}", random.choice(CATEGORIES),
             round(random.uniform(5, 2000), 2), random.randint(0, 500))
            for i in range(1, n+1)]

def make_orders(n, max_cust, max_prod):
    base = datetime.date(2023, 1, 1)
    return [(i, random.randint(1, max_cust), random.randint(1, max_prod),
             random.randint(1, 10), round(random.uniform(10, 5000), 2),
             (base + datetime.timedelta(days=random.randint(0, 365))).isoformat())
            for i in range(1, n+1)]

cust_schema   = ["customer_id","name","country","status","lifetime_value"]
prod_schema   = ["product_id","name","category","price","stock"]
order_schema  = ["order_id","customer_id","product_id","quantity","amount","order_date"]

# Three sizes: small (lookup), medium, large (fact)
customers_sm  = spark.createDataFrame(make_customers(1_000),   cust_schema)   # 1K
customers_md  = spark.createDataFrame(make_customers(50_000),  cust_schema)   # 50K
products_sm   = spark.createDataFrame(make_products(500),      prod_schema)   # 500
orders_lg     = spark.createDataFrame(make_orders(500_000, 50_000, 500), order_schema)  # 500K

customers_sm.cache().count()
customers_md.cache().count()
products_sm.cache().count()
orders_lg.cache().count()

print("Datasets ready:")
print(f"  customers_sm : {customers_sm.count():>8,} rows")
print(f"  customers_md : {customers_md.count():>8,} rows")
print(f"  products_sm  : {products_sm.count():>8,} rows")
print(f"  orders_lg    : {orders_lg.count():>8,} rows")

[Stage 0:>                                                          (0 + 0) / 2]

## Step 2 — BroadcastHashJoin: The Fastest Join

**How it works:**
1. Small table is collected to driver
2. Driver broadcasts it to ALL executors
3. Each executor builds a hash map from the small table in memory
4. Large table scans locally — for each row, hash lookup in the map
5. **No shuffle required** — this is why it's fast

**When Spark chooses it automatically:**
`spark.sql.autoBroadcastJoinThreshold = 10MB` (default)
If one side is < 10 MB, Spark broadcasts it automatically.

**When to force it:**
When your table is > 10 MB but you know it fits in executor memory.


In [ ]:
# Check auto-broadcast threshold
threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
print(f"Auto-broadcast threshold: {threshold} bytes ({int(threshold)//1024//1024} MB)")
print()

# 1. Auto BroadcastHashJoin (products_sm is small enough)
auto_broadcast = orders_lg.join(products_sm, "product_id")
plan = auto_broadcast._jdf.queryExecution().executedPlan().toString()
strategy = "BroadcastHashJoin" if "BroadcastHashJoin" in plan else "other"
print(f"Auto join on products_sm (500 rows): {strategy}")

# 2. Force BroadcastHashJoin with hint even for larger table
forced_broadcast = orders_lg.join(F.broadcast(customers_md), "customer_id")
plan2 = forced_broadcast._jdf.queryExecution().executedPlan().toString()
strategy2 = "BroadcastHashJoin" if "BroadcastHashJoin" in plan2 else "other"
print(f"Forced broadcast on customers_md (50K rows): {strategy2}")

# Benchmark
def time_join(df, label, runs=3):
    times = []
    for _ in range(runs):
        t0 = time.time()
        df.agg(F.sum("amount"), F.count("*")).collect()
        times.append(time.time() - t0)
    median = sorted(times)[1]
    print(f"  {label:<45} {median:.3f}s")
    return median

print()
print("Join benchmarks (orders 500K × customers 50K):")
t_broadcast = time_join(
    orders_lg.join(F.broadcast(customers_md), "customer_id"),
    "BroadcastHashJoin (forced)"
)
t_smj = time_join(
    orders_lg.hint("merge").join(customers_md, "customer_id"),
    "SortMergeJoin (forced)"
)
print(f"  BroadcastHashJoin speedup: {t_smj/t_broadcast:.1f}x faster")

In [ ]:
# Show the plans side by side
print("BroadcastHashJoin plan:")
orders_lg.join(F.broadcast(customers_md), "customer_id").explain(mode="formatted")

print("\nSortMergeJoin plan:")
orders_lg.hint("merge").join(customers_md, "customer_id").explain(mode="formatted")

## Step 3 — SortMergeJoin: The Default for Large Tables

**How it works:**
1. Both sides are shuffled by join key (Exchange nodes)
2. Both sides are sorted by join key
3. Two sorted streams are merged (like merge in merge-sort)
4. Matching rows are joined as the merge advances

**When to use it:**
- Both tables are large (> autoBroadcastJoinThreshold)
- Neither fits in executor memory for a hash map

**Cost:**
- 2 shuffles (one per table)
- 2 sorts
- This is why SortMergeJoin is the most expensive join type


In [ ]:
# Disable auto-broadcast to force SortMergeJoin
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

smj = orders_lg.join(customers_md, "customer_id")
plan = smj._jdf.queryExecution().executedPlan().toString()
strategy = "SortMergeJoin" if "SortMergeJoin" in plan else plan[:50]
print(f"Join strategy with broadcast disabled: {strategy}")
print()
print("SortMergeJoin physical plan:")
smj.explain(mode="formatted")

# Re-enable
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

## Step 4 — ShuffledHashJoin: No Sort Needed

**How it works:**
1. Smaller side is shuffled and a **hash map** is built per partition
2. Larger side is shuffled to matching partitions
3. Each partition does a hash lookup (no sorting)

**Faster than SortMergeJoin** because it avoids sorting.
But it **requires enough memory** to build the hash map for the smaller side.
If the smaller side is too large → spill → worse than SortMergeJoin.

**When Spark chooses it:**
Only when `spark.sql.join.preferSortMergeJoin = false` (default: true)
or when the `shuffle_hash` hint is used.


In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# Force ShuffledHashJoin
shj = orders_lg.hint("shuffle_hash").join(customers_md, "customer_id")
plan = shj._jdf.queryExecution().executedPlan().toString()
strategy = ("ShuffledHashJoin" if "ShuffledHashJoin" in plan
            else "SortMergeJoin" if "SortMergeJoin" in plan else "other")
print(f"With shuffle_hash hint: {strategy}")

print()
print("Benchmark: SortMergeJoin vs ShuffledHashJoin (both sides large, no broadcast):")
t_smj = time_join(
    orders_lg.hint("merge").join(customers_md, "customer_id"),
    "SortMergeJoin"
)
t_shj = time_join(
    orders_lg.hint("shuffle_hash").join(customers_md, "customer_id"),
    "ShuffledHashJoin"
)
print(f"\nShuffledHashJoin {'faster' if t_shj < t_smj else 'slower'}: {abs(t_smj/t_shj - 1)*100:.0f}%")
print("(ShuffledHashJoin wins when smaller side fits in executor memory)")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))

## Step 5 — Skewed Joins: The Silent Killer

**Data skew** happens when one or few join keys have vastly more rows than others.
The executor handling the hot key takes 10x-100x longer than others.

**Symptoms:**
- One task takes much longer than all others (check Spark UI Stages)
- Job is "almost done" for a long time
- One executor at 100% CPU while others are idle

**AQE Skew Join** automatically detects and splits skewed partitions.
Let's observe it in action.


In [ ]:
# Create a severely skewed dataset
# customer_id=1 has 80% of all orders
random.seed(42)
N = 200_000

skewed_orders = spark.createDataFrame([
    (i,
     1 if random.random() < 0.8 else random.randint(2, 1000),  # 80% to customer 1
     random.randint(1, 500),
     random.randint(1, 10),
     round(random.uniform(10, 500), 2),
     "2024-01-01")
    for i in range(1, N+1)
], order_schema)

# Check skew
print("Key distribution (top 5):")
skewed_orders.groupBy("customer_id") \
             .count() \
             .orderBy(F.desc("count")) \
             .show(5)

skewed_count = skewed_orders.filter(F.col("customer_id") == 1).count()
print(f"customer_id=1 has {skewed_count:,} / {N:,} rows ({skewed_count/N*100:.0f}%)")

In [ ]:
# Join WITH AQE skew handling (enabled by default)
print("AQE settings:")
print(f"  adaptive.enabled           : {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"  skewJoin.enabled           : {spark.conf.get('spark.sql.adaptive.skewJoin.enabled')}")
print(f"  skewedPartitionFactor      : {spark.conf.get('spark.sql.adaptive.skewJoin.skewedPartitionFactor', '5')}")
print()

# WITH AQE skew join
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
t0 = time.time()
r_aqe = skewed_orders.join(customers_md, "customer_id") \
                     .groupBy("country").agg(F.sum("amount")).collect()
t_aqe = time.time() - t0

# WITHOUT AQE skew join
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")
t0 = time.time()
r_no_aqe = skewed_orders.join(customers_md, "customer_id") \
                        .groupBy("country").agg(F.sum("amount")).collect()
t_no_aqe = time.time() - t0

spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

print(f"With AQE skew join    : {t_aqe:.3f}s")
print(f"Without AQE skew join : {t_no_aqe:.3f}s")
print(f"AQE speedup           : {t_no_aqe/t_aqe:.1f}x")
print()
print("AQE detected the skew on customer_id=1 and split its partition")
print("into multiple sub-partitions, distributing work evenly.")

## Step 6 — Manual Skew Handling: Salting

When AQE cannot fix the skew (e.g., in older Spark or specific cases),
**salting** is the manual solution:

1. Add a random salt (0 to N) to the hot key on the left side
2. Explode the right side to have N copies of the hot key rows
3. Join on (key, salt) — now the hot key is split across N partitions


In [ ]:
SALT_FACTOR = 4  # split hot key across 4 partitions

# Left side: add salt column
salted_orders = skewed_orders.withColumn(
    "salt", (F.rand(42) * SALT_FACTOR).cast("int")
).withColumn(
    "salted_key", F.concat_ws("_", F.col("customer_id").cast("string"), F.col("salt"))
)

# Right side: replicate each row SALT_FACTOR times with each salt value
salt_df = spark.range(SALT_FACTOR).select(F.col("id").alias("salt"))
salted_customers = customers_md.crossJoin(F.broadcast(salt_df)) \
    .withColumn(
        "salted_key",
        F.concat_ws("_", F.col("customer_id").cast("string"), F.col("salt"))
    )

print(f"Orders with salt     : {salted_orders.count():,} rows")
print(f"Customers replicated : {salted_customers.count():,} rows ({customers_md.count():,} × {SALT_FACTOR})")

# Join on salted key
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")
t0 = time.time()
r_salted = salted_orders.join(salted_customers, "salted_key") \
                        .groupBy("country").agg(F.sum("amount")).collect()
t_salted = time.time() - t0

print(f"\nSalted join time  : {t_salted:.3f}s")
print(f"No-AQE naive time : {t_no_aqe:.3f}s")
print(f"Salting speedup   : {t_no_aqe/t_salted:.1f}x")
print()
print("Trade-off: salting replicates the right side SALT_FACTOR times.")
print("Use AQE when available — salting is the fallback for older Spark.")

spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

## Step 7 — Join Hints Cheat Sheet

```python
# BroadcastHashJoin — no shuffle, fastest for small tables
df1.join(F.broadcast(df2), "key")
df1.hint("broadcast").join(df2, "key")

# SortMergeJoin — safe for any size, 2 shuffles + 2 sorts
df1.hint("merge").join(df2, "key")

# ShuffledHashJoin — 1 shuffle, no sort, needs memory for hash map
df1.hint("shuffle_hash").join(df2, "key")

# ShuffleReplicateNestedLoopJoin — non-equi joins
df1.hint("shuffle_replicate_nl").join(df2, df1.key > df2.key)
```

## Summary: Join Strategy Decision Guide

```
Table sizes?
  One side < 10 MB (autoBroadcastJoinThreshold)
      → BroadcastHashJoin (automatic)
  One side < 200 MB and fits in executor memory
      → F.broadcast() hint or increase threshold
  Both sides large, enough memory for hash map
      → hint("shuffle_hash")  [faster than SMJ, needs memory]
  Both sides large, memory constrained
      → SortMergeJoin (default, safe, predictable)

Data skew detected?
  AQE enabled (default)
      → AQE handles it automatically (skewJoin.enabled=true)
  AQE not available / not sufficient
      → Manual salting
```

| Metric | BroadcastHashJoin | ShuffledHashJoin | SortMergeJoin |
|---|---|---|---|
| Shuffle | None | 1 (smaller side) | 2 (both sides) |
| Sort | None | None | 2 (both sides) |
| Memory | Broadcast size | Hash map (smaller side) | Sort buffers |
| Best for | Small + Large | Medium + Medium | Large + Large |
| Spill risk | None | Medium | Low |
